In [1]:
from datasets import load_dataset
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
)
from transformers.models import Wav2Vec2Config

from yohane_fa.model import YohaneForcedAligner

In [2]:
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True,
)
tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(
    "./",
    word_delimiter_token="|",
    unk_token="[UNK]",
    pad_token="[PAD]",
)
processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)
processor

Wav2Vec2Processor:
- feature_extractor: Wav2Vec2FeatureExtractor {
  "do_normalize": true,
  "feature_extractor_type": "Wav2Vec2FeatureExtractor",
  "feature_size": 1,
  "padding_side": "right",
  "padding_value": 0.0,
  "return_attention_mask": true,
  "sampling_rate": 16000
}

- tokenizer: Wav2Vec2CTCTokenizer(name_or_path='./', vocab_size=30, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '[UNK]', 'pad_token': '[PAD]', 'word_delimiter_token': '|'}, added_tokens_decoder={
	0: AddedToken("|", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	28: AddedToken("[UNK]", rstrip=True, lstrip=True, single_word=False, normalized=False, special=False),
	29: AddedToken("[PAD]", rstrip=True, lstrip=True, single_word=False, normalized=False, special=False),
	30: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, speci

In [3]:
model = YohaneForcedAligner(
    Wav2Vec2Config(
        vocab_size=processor.tokenizer.vocab_size,
        hidden_size=256,
        num_hidden_layers=10,
        num_attention_heads=4,
        hidden_dropout=0.0,
        activation_dropout=0.0,
        attention_dropout=0.0,
        feat_proj_dropout=0.0,
        feat_quantizer_dropout=0.0,
        final_dropout=0.0,
        intermediate_size=680,
        conv_dim=[256, 256, 256, 256, 256, 256, 256],
        conv_stride=[5, 2, 2, 2, 2, 2, 2],
        do_stable_layer_norm=True,
    )
)
model

YohaneForcedAligner(
  (wav2vec2): CustomWav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2GroupNormConvLayer(
          (conv): Conv1d(1, 256, kernel_size=(10,), stride=(5,), bias=False)
          (activation): GELUActivation()
          (layer_norm): GroupNorm(256, 256, eps=1e-05, affine=True, bias=True)
        )
        (1-4): 4 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(256, 256, kernel_size=(3,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(256, 256, kernel_size=(2,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): RMSNormWav2Vec2FeatureProjection(
      (layer_norm): RMSNorm((256,), eps=1e-05, elementwise_affine=True)
      (projection): Linear(in_features=256, out_features=256, bias=False)
      (dropout): Dropout(p=0.0, inplac

In [4]:
dataset = load_dataset("NextFire/karaoke-mugen-timings", split="train", streaming=True)
dataset

Resolving data files:   0%|          | 0/189 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/189 [00:00<?, ?it/s]

IterableDataset({
    features: ['audio', 'metadata', 'subtitles', 'timings'],
    num_shards: 189
})

In [5]:
for example in dataset:
    output = processor(
        audio=example["audio"]["array"],
        sampling_rate=example["audio"]["sampling_rate"],  # pyright: ignore[reportCallIssue]
        text="hello",
    )
    break

output  # pyright: ignore[reportPossiblyUnboundVariable]

{'input_values': [array([-0.00311542, -0.00481431, -0.00354014, ...,  0.00070709,
        0.00070709,  0.00070709], shape=(1439403,), dtype=float32)], 'attention_mask': [array([1, 1, 1, ..., 1, 1, 1], shape=(1439403,), dtype=int32)], 'labels': [8, 5, 12, 12, 15]}